
# Regression Analysis

## Project: Delivery Time Deviation Prediction in Logistics

This notebook performs regression analysis for the logistics delivery dataset.

Main steps:
1. Define dependent variable `Y`.
2. Define independent variables `X`.
3. Split data into training and testing sets.
4. Train regression models.
5. Predict on the test set.
6. Evaluate model performance using R-squared, MAE, RMSE, and MAPE.

For this project:
- Dependent variable `Y`: `delivery_time_deviation`
- Independent variables `X`: logistics-related features such as traffic congestion, ETA variation, route risk, weather severity, driver behavior, delay probability, and engineered features.


## 1. Import Libraries and Load Dataset

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)

cleaned_file = Path("cleaned_dynamic_supply_chain_logistics_dataset.csv")
original_file = Path("dynamic_supply_chain_logistics_dataset.csv")

if cleaned_file.exists():
    df = pd.read_csv(cleaned_file)
    print("Loaded cleaned dataset.")
elif original_file.exists():
    df = pd.read_csv(original_file)
    print("Loaded original dataset.")
else:
    raise FileNotFoundError("Dataset file not found. Please put the CSV file in the same folder as this notebook.")

print("Dataset shape:", df.shape)
display(df.head())


## 2. Data Preparation for Regression

In [ ]:

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df.sort_values("timestamp").reset_index(drop=True)

target_col = "delivery_time_deviation"

df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["is_peak_hour"] = df["hour_of_day"].isin([7, 8, 9, 17, 18, 19]).astype(int)

df["is_late"] = (df[target_col] > 0).astype(int)
df["distance_proxy"] = df["eta_variation_hours"]

df["vehicle_score_proxy"] = (
    df["driver_behavior_score"] * 0.6 +
    (100 - df["fuel_consumption_rate"]) * 0.4
)

df["lat_bin"] = pd.cut(df["vehicle_gps_latitude"], bins=5, labels=False)
df["long_bin"] = pd.cut(df["vehicle_gps_longitude"], bins=5, labels=False)
df["gps_zone"] = df["lat_bin"].astype(str) + "_" + df["long_bin"].astype(str)

df["final_delivery_time_hours"] = (
    df["lead_time_days"] * 24
    + df["loading_unloading_time"]
    + df["customs_clearance_time"]
    + df["eta_variation_hours"]
)

df["traffic_intensity_proxy"] = df["traffic_congestion_level"] * df["is_peak_hour"]

df["route_complexity"] = (
    df["traffic_congestion_level"]
    + df["route_risk_level"]
    + df["weather_condition_severity"] * 10
) / 3

df["traffic_x_weather"] = df["traffic_congestion_level"] * df["weather_condition_severity"]
df["traffic_x_route_risk"] = df["traffic_congestion_level"] * df["route_risk_level"]
df["eta_variation_x_peak_hour"] = df["eta_variation_hours"] * df["is_peak_hour"]

display(df[[
    "timestamp",
    "hour_of_day",
    "distance_proxy",
    "vehicle_score_proxy",
    "traffic_intensity_proxy",
    "route_complexity",
    target_col
]].head())


## 3. Lag and Rolling Average Feature Engineering

In [ ]:

df["lag_delivery_deviation_1"] = df[target_col].shift(1)
df["lag_delivery_deviation_3"] = df[target_col].shift(3)
df["lag_delivery_deviation_7"] = df[target_col].shift(7)

df["rolling_avg_deviation_3"] = df[target_col].shift(1).rolling(window=3).mean()
df["rolling_avg_deviation_7"] = df[target_col].shift(1).rolling(window=7).mean()
df["rolling_avg_deviation_14"] = df[target_col].shift(1).rolling(window=14).mean()

df_model = df.dropna().reset_index(drop=True)

print("Shape after lag and rolling features:", df_model.shape)

display(df_model[[
    target_col,
    "lag_delivery_deviation_1",
    "lag_delivery_deviation_3",
    "rolling_avg_deviation_3",
    "rolling_avg_deviation_7"
]].head())



## 4. Define Dependent Variable Y and Independent Variables X

- `Y` is the dependent variable that we want to predict.
- `X` contains independent variables used to predict `Y`.


In [ ]:

feature_cols = [
    "vehicle_gps_latitude",
    "vehicle_gps_longitude",
    "fuel_consumption_rate",
    "eta_variation_hours",
    "traffic_congestion_level",
    "warehouse_inventory_level",
    "loading_unloading_time",
    "handling_equipment_availability",
    "order_fulfillment_status",
    "weather_condition_severity",
    "port_congestion_level",
    "shipping_costs",
    "supplier_reliability_score",
    "lead_time_days",
    "historical_demand",
    "iot_temperature",
    "cargo_condition_status",
    "route_risk_level",
    "customs_clearance_time",
    "driver_behavior_score",
    "fatigue_monitoring_score",
    "disruption_likelihood_score",
    "delay_probability",
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "is_peak_hour",
    "distance_proxy",
    "vehicle_score_proxy",
    "final_delivery_time_hours",
    "traffic_intensity_proxy",
    "route_complexity",
    "traffic_x_weather",
    "traffic_x_route_risk",
    "eta_variation_x_peak_hour",
    "lag_delivery_deviation_1",
    "lag_delivery_deviation_3",
    "lag_delivery_deviation_7",
    "rolling_avg_deviation_3",
    "rolling_avg_deviation_7",
    "rolling_avg_deviation_14"
]

X = df_model[feature_cols]
Y = df_model[target_col]

print("Independent variables X shape:", X.shape)
print("Dependent variable Y shape:", Y.shape)
display(X.head())


## 5. Time-Based Train-Test Split

In [ ]:

split_index = int(len(df_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

Y_train = Y.iloc[:split_index]
Y_test = Y.iloc[split_index:]

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)


## 6. Define Evaluation Metrics

In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calculate_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    epsilon = 1e-8
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + epsilon))) * 100

    return r2, mae, rmse, mape


## 7. Train Regression Models

In [ ]:

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=8),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        objective="reg:squarederror"
    )
except ImportError:
    print("XGBoost is not installed. Run: pip install xgboost")


In [ ]:

model_results = []
predictions = {}
trained_models = {}

for model_name, model in models.items():
    print("Training:", model_name)

    model.fit(X_train, Y_train)
    y_pred = model.predict(X_test)

    r2, mae, rmse, mape = calculate_metrics(Y_test, y_pred)

    model_results.append({
        "Model": model_name,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE (%)": mape
    })

    predictions[model_name] = y_pred
    trained_models[model_name] = model

results_df = pd.DataFrame(model_results).sort_values(by="MAPE (%)")

display(results_df)


## 8. Model Comparison Visualization

In [ ]:

plt.figure(figsize=(10, 5))

sns.barplot(
    data=results_df,
    x="Model",
    y="MAPE (%)"
)

plt.title("Regression Model Comparison Based on MAPE")
plt.xlabel("Model")
plt.ylabel("MAPE (%)")
plt.xticks(rotation=30)
plt.show()


## 9. Linear Regression Coefficients

In [ ]:

if "Linear Regression" in trained_models:
    linear_model = trained_models["Linear Regression"]

    coef_df = pd.DataFrame({
        "Feature": feature_cols,
        "Coefficient": linear_model.coef_
    }).sort_values(by="Coefficient", ascending=False)

    print("Top positive coefficients:")
    display(coef_df.head(15))

    print("Top negative coefficients:")
    display(coef_df.tail(15))


## 10. Actual vs Predicted Plot

In [ ]:

best_model_name = results_df.iloc[0]["Model"]
best_pred = predictions[best_model_name]

print("Best model based on MAPE:", best_model_name)

comparison_df = pd.DataFrame({
    "Actual": Y_test.values,
    "Predicted": best_pred,
    "Error": Y_test.values - best_pred
})

display(comparison_df.head(20))

plt.figure(figsize=(8, 6))

plt.scatter(Y_test, best_pred, alpha=0.4)
plt.xlabel("Actual Delivery Time Deviation")
plt.ylabel("Predicted Delivery Time Deviation")
plt.title(f"Actual vs Predicted - {best_model_name}")
plt.grid(True, alpha=0.3)
plt.show()


## 11. Feature Importance

In [ ]:

best_model = trained_models[best_model_name]

if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": feature_cols,
        "Importance": best_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    display(importance_df.head(15))

    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=importance_df.head(15),
        x="Importance",
        y="Feature"
    )

    plt.title(f"Top 15 Feature Importance - {best_model_name}")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.show()
else:
    print("The best model does not provide feature importance.")


## 12. Export Regression Results

In [ ]:

results_df.to_csv("regression_model_comparison_results.csv", index=False)

prediction_output = df_model.iloc[split_index:].copy()
prediction_output["best_model_prediction"] = best_pred
prediction_output["prediction_error"] = prediction_output[target_col] - prediction_output["best_model_prediction"]

prediction_output.to_csv("regression_prediction_results.csv", index=False)

print("Exported files:")
print("- regression_model_comparison_results.csv")
print("- regression_prediction_results.csv")



## 13. Summary

This regression analysis completed the following tasks:

1. Defined `delivery_time_deviation` as the dependent variable `Y`.
2. Selected logistics-related features as independent variables `X`.
3. Created time-based, proxy, interaction, lag, and rolling average features.
4. Split the dataset using a time-based train-test split.
5. Trained five regression models:
   - Linear Regression
   - Ridge Regression
   - Decision Tree Regressor
   - Random Forest Regressor
   - XGBoost Regressor
6. Evaluated the models using R-squared, MAE, RMSE, and MAPE.
7. Compared model performance and selected the best model.
8. Visualized actual vs predicted values.
9. Analyzed feature importance for interpretable tree-based models.
